incremental data

In [0]:
# ============================================================
# BRONZE LAYER - ZERO HARDCODING (All from Config/Secrets)
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
from datetime import datetime

# ============================================================
# GET EVERYTHING FROM CONFIGURATION (No hardcoding!)
# ============================================================

# 1. Environment from ADF widget
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from ADF: {env}")
except:
    env = "DEV"
    print(f"📌 Default environment: {env}")

# 2. Storage account from Spark Config (set in Comput
storage_account_name = "capstonestorage01"  # This is fine as it's not a secret

# 3. Access key from Spark Config (already set in Compute!)
# No need to get it - Spark already has it configured!

# 4. Container names based on environment only
if env == 'DEV':
    bronze_container = 'bronze-dev'
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    bronze_container = 'bronze-test'
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    bronze_container = 'bronze'
    silver_container = 'silver'
    gold_container = 'gold'

# 5. RAW container is FIXED (only one)
raw_container = "raw"

# Build paths
RAW = f"abfss://{raw_container}@{storage_account_name}.dfs.core.windows.net"
BRONZE = f"abfss://{bronze_container}@{storage_account_name}.dfs.core.windows.net"
GOLD = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 RAW: {raw_container}")
print(f"📁 BRONZE: {bronze_container}")
print(f"📍 BRONZE Path: {BRONZE}")
print(f"{'='*55}\n")

# ============================================================
# WATERMARK PARAMETER
# ============================================================
dbutils.widgets.text("last_load_date", "2019-12-31")
last_load_date = dbutils.widgets.get("last_load_date")
print(f"📅 Watermark: {last_load_date}")

# ============================================================
# AUDIT LOG FUNCTION
# ============================================================
def log_audit(layer, table, status, row_count=0, message=""):
    try:
        spark.createDataFrame([{
            "layer": layer,
            "table_name": table,
            "status": status,
            "row_count": row_count,
            "message": message,
            "environment": env,
            "processed_at": str(datetime.now())
        }]).write.format("delta").mode("append").save(f"{GOLD}/audit_log")
    except Exception as e:
        print(f"  ⚠️ Audit error: {e}")
    print(f"  📋 AUDIT | {status} | {table} | rows: {row_count:,}")

# ============================================================
# MAIN PROCESSING
# ============================================================
try:
    ingestion_time = F.lit(datetime.now()).cast(TimestampType())

    # Master Files (Full Overwrite)
    print("\n📁 Loading Master Files...")

    for master_file in ["customer_master", "account_master", "branch_channel", "repayment_delinquency"]:
        df = spark.read.option("header", "true").option("inferSchema", "true") \
            .csv(f"{RAW}/{master_file}.csv") \
            .withColumn("ingestion_timestamp", ingestion_time) \
            .withColumn("source_file", F.lit(f"{master_file}.csv"))
        
        df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(f"{BRONZE}/{master_file}")
        log_audit("bronze", master_file, "success", df.count())

    # Transactions (Incremental)
    print(f"\n📁 Loading Transactions AFTER {last_load_date}...")

    df_all = spark.read.option("header", "true").option("inferSchema", "true") \
        .csv(f"{RAW}/transactions/*.csv")

    if "ingestion_timestamp" in df_all.columns:
        df_all = df_all.withColumnRenamed("ingestion_timestamp", "source_ingestion_timestamp")

    # Filter by watermark
    df_filtered = df_all.filter(F.col("transaction_date") > last_load_date)
    new_count = df_filtered.count()
    
    if new_count > 0:
        df_filtered = df_filtered \
            .withColumn("year", F.year("transaction_date")) \
            .withColumn("month", F.month("transaction_date")) \
            .withColumn("ingestion_timestamp", ingestion_time)
        
        df_filtered.write.format("delta").mode("append") \
            .partitionBy("product_type", "year") \
            .save(f"{BRONZE}/transactions")
        
        log_audit("bronze", "transactions", "success", new_count)
    else:
        print("  No new transactions")
        log_audit("bronze", "transactions", "success", 0, "No new data")

    # Summary
    print("\n" + "=" * 55)
    print(f"  BRONZE SUMMARY - {env}")
    print("=" * 55)
    for table in ["customer_master", "account_master", "branch_channel", "repayment_delinquency", "transactions"]:
        try:
            count = spark.read.format("delta").load(f"{BRONZE}/{table}").count()
            print(f"  ✅ {table:<25} {count:>10,} rows")
        except:
            print(f"  ⚠️ {table:<25} Not found")
    print("=" * 55)
    print(f"  ✅ BRONZE COMPLETE! ({env})")
    print("=" * 55)

except Exception as e:
    log_audit("bronze", "transactions", "failed", message=str(e))
    raise